# Proyecto Estadísticas de YouTube: Avances IV (producto final)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dairo123e/Proyecto-Estadisticas-YouTube/blob/main/Proyecto_Estadisticas_YouTube_Avances_IV.ipynb)

**Integrantes:** Dairo Enrique Contreras Quintana · Stiven David Álvarez Olmos
**Docente:** Paul Alexander Díaz Montaña
**Curso:** Procesamiento de lenguaje natural, Especialización en Inteligencia Artificial, Universidad de Cundinamarca

Este notebook consolida el proyecto completo y entrega el producto final de la semana 10:

1. Carga de datos directamente desde el repositorio de GitHub.
2. Limpieza, integración y construcción de variables (Avances I y II).
3. Entrenamiento de los dos modelos definitivos seleccionados en el Avance III.
4. Evaluación final y comparación con el diseño inicial.
5. Persistencia de modelos, función de predicción, interfaz web y API con Gradio.
6. Monitoreo básico: registro de predicciones y detección de deriva de datos.

## 1. Instalación y librerías

In [ ]:
import sys, subprocess
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio", "xgboost", "lightgbm"], check=True)
print("Ejecutando en Google Colab:", EN_COLAB)

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from datetime import datetime

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, roc_curve, ConfusionMatrixDisplay)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
SEMILLA = 42
print("Librerías cargadas correctamente")

## 2. Carga de datos desde GitHub
Los archivos originales del dataset *YouTube Statistics* de Kaggle se leen desde el repositorio, de modo que el notebook se ejecuta en Colab sin subir archivos manualmente.

In [ ]:
URL_BASE = "https://raw.githubusercontent.com/dairo123e/Proyecto-Estadisticas-YouTube/main/"
def leer(nombre):
    return pd.read_csv(nombre) if os.path.exists(nombre) else pd.read_csv(URL_BASE + nombre)

videos = leer("videos-stats.csv")
comentarios = leer("comments.csv")
print("Videos:", videos.shape, "| Comentarios:", comentarios.shape)
videos.head(3)

## 3. Limpieza e integración (proceso de los Avances I y II)

In [ ]:
videos = videos.drop(columns=["Unnamed: 0"], errors="ignore")
comentarios = comentarios.drop(columns=["Unnamed: 0"], errors="ignore")
videos["Published At"] = pd.to_datetime(videos["Published At"], errors="coerce")
for c in ["Likes", "Comments", "Views"]:
    videos.loc[videos[c] < 0, c] = np.nan
videos = videos.dropna(subset=["Likes", "Comments", "Views"]).drop_duplicates()
comentarios = comentarios.dropna(subset=["Comment"])
comentarios = comentarios[comentarios["Comment"].astype(str).str.strip() != ""].drop_duplicates()

for etiqueta, valor in [("Negativo", 0), ("Neutral", 1), ("Positivo", 2)]:
    comentarios[etiqueta] = (comentarios["Sentiment"] == valor).astype(int)

resumen = comentarios.groupby("Video ID").agg(
    comentarios_analizados=("Comment", "count"),
    promedio_likes_comentarios=("Likes", "mean"),
    sentimiento_promedio=("Sentiment", "mean"),
    negativos=("Negativo", "sum"), neutrales=("Neutral", "sum"), positivos=("Positivo", "sum"),
    texto_comentarios=("Comment", lambda s: " ".join(s.astype(str)))
).reset_index()
for c, n in [("negativos", "proporcion_negativos"), ("neutrales", "proporcion_neutrales"), ("positivos", "proporcion_positivos")]:
    resumen[n] = resumen[c] / resumen["comentarios_analizados"]

# Un registro por video: algunos videos aparecen en varias palabras clave
videos_unicos = videos.sort_values("Video ID").drop_duplicates(subset=["Video ID"], keep="first")
datos = videos_unicos.merge(resumen, on="Video ID", how="inner").reset_index(drop=True)
print("Videos integrados:", len(datos))

## 4. Variable objetivo y variables predictoras

In [ ]:
MEDIANA_VISTAS = datos["Views"].median()
datos["popularidad_alta"] = (datos["Views"] > MEDIANA_VISTAS).astype(int)
FECHA_REFERENCIA = datos["Published At"].max()
datos["Title"] = datos["Title"].astype(str)
datos["longitud_titulo"] = datos["Title"].str.len()
datos["antiguedad_dias"] = (FECHA_REFERENCIA - datos["Published At"]).dt.days
datos["log_likes_video"] = np.log1p(datos["Likes"])
datos["log_comentarios_video"] = np.log1p(datos["Comments"])
datos["log_promedio_likes_comentarios"] = np.log1p(datos["promedio_likes_comentarios"])
datos["mayusculas_titulo"] = datos["Title"].apply(lambda t: sum(c.isupper() for c in t) / max(len(t), 1))
datos["signos_titulo"] = datos["Title"].str.count(r"[!?]")
datos["palabras_titulo"] = datos["Title"].str.split().str.len()
datos["titulo_numero"] = datos["Title"].str.contains(r"\d").astype(int)
datos["longitud_comentarios"] = datos["texto_comentarios"].str.len() / datos["comentarios_analizados"]

print("Mediana de visualizaciones:", MEDIANA_VISTAS)
print(datos["popularidad_alta"].value_counts().rename({0: "Baja", 1: "Alta"}))

In [ ]:
# Modelo A (principal): variables de interacción + sentimiento (Avances II y III)
VARS_A = ["log_likes_video", "log_comentarios_video", "log_promedio_likes_comentarios",
          "comentarios_analizados", "proporcion_negativos", "proporcion_neutrales",
          "proporcion_positivos", "longitud_titulo", "antiguedad_dias"]
# Modelo B (NLP): sin likes ni comentarios del video, con texto TF-IDF de títulos y comentarios
VARS_B = ["log_promedio_likes_comentarios", "comentarios_analizados", "proporcion_negativos",
          "proporcion_neutrales", "proporcion_positivos", "longitud_titulo", "antiguedad_dias",
          "mayusculas_titulo", "signos_titulo", "palabras_titulo", "titulo_numero", "longitud_comentarios"]
CATEGORICAS = ["Keyword"]
TEXTO = ["Title", "texto_comentarios"]

y = datos["popularidad_alta"]
X_train, X_test, y_train, y_test = train_test_split(
    datos, y, test_size=0.2, stratify=y, random_state=SEMILLA)
print("Entrenamiento:", X_train.shape[0], "| Prueba:", X_test.shape[0])

## 5. Entrenamiento de los modelos definitivos
Los hiperparámetros provienen de la búsqueda aleatoria con validación cruzada del Avance III.

In [ ]:
def preprocesador(numericas, con_texto=False):
    partes = [
        ("num", Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler())]), numericas),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS)]
    if con_texto:
        def bloque(k, maxf):
            return Pipeline([("tfidf", TfidfVectorizer(max_features=maxf, ngram_range=(1, 2), min_df=3,
                                                         sublinear_tf=True, stop_words="english")),
                             ("svd", TruncatedSVD(k, random_state=SEMILLA))])
        partes += [("titulo", bloque(20, 2000), "Title"), ("comentarios", bloque(30, 5000), "texto_comentarios")]
    return ColumnTransformer(partes)

modelo_A = Pipeline([("prep", preprocesador(VARS_A)),
                     ("modelo", GradientBoostingClassifier(n_estimators=300, learning_rate=0.1, max_depth=3,
                                                           min_samples_leaf=5, subsample=1.0, random_state=SEMILLA))])
modelo_B = Pipeline([("prep", preprocesador(VARS_B, con_texto=True)),
                     ("modelo", XGBClassifier(n_estimators=400, learning_rate=0.02, max_depth=4, subsample=0.7,
                                              colsample_bytree=1.0, reg_lambda=5, eval_metric="logloss",
                                              random_state=SEMILLA, verbosity=0))])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
resultados = []
for nombre, modelo in [("A. Gradient Boosting ajustado (con likes)", modelo_A),
                       ("B. XGBoost + TF-IDF (sin likes)", modelo_B)]:
    t0 = time.time(); modelo.fit(X_train, y_train); t_ent = time.time() - t0
    pred = modelo.predict(X_test); prob = modelo.predict_proba(X_test)[:, 1]
    cv_f1 = cross_val_score(modelo, X_train, y_train, cv=cv, scoring="f1")
    resultados.append({"Modelo": nombre, "Accuracy": accuracy_score(y_test, pred),
                       "Precision": precision_score(y_test, pred), "Recall": recall_score(y_test, pred),
                       "F1": f1_score(y_test, pred), "ROC_AUC": roc_auc_score(y_test, prob),
                       "CV_F1": cv_f1.mean(), "CV_std": cv_f1.std(), "Entrenamiento_s": t_ent})
resultados = pd.DataFrame(resultados)
resultados.round(4)

## 6. Evaluación final

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(16, 4.6))
for eje, (nombre, modelo) in zip(ejes[:2], [("Modelo A (con likes)", modelo_A), ("Modelo B (sin likes, NLP)", modelo_B)]):
    ConfusionMatrixDisplay(confusion_matrix(y_test, modelo.predict(X_test)),
                           display_labels=["Baja", "Alta"]).plot(ax=eje, cmap="Blues", colorbar=False)
    eje.set_title(nombre); eje.grid(False)
for nombre, modelo in [("Modelo A", modelo_A), ("Modelo B", modelo_B)]:
    p = modelo.predict_proba(X_test)[:, 1]; fpr, tpr, _ = roc_curve(y_test, p)
    ejes[2].plot(fpr, tpr, label=f"{nombre} (AUC = {roc_auc_score(y_test, p):.3f})")
ejes[2].plot([0, 1], [0, 1], "--", color="gray"); ejes[2].legend(); ejes[2].set_title("Curvas ROC")
ejes[2].set_xlabel("Tasa de falsos positivos"); ejes[2].set_ylabel("Tasa de verdaderos positivos")
plt.tight_layout(); plt.savefig("evaluacion_final.png", dpi=150); plt.show()

In [ ]:
# Evolución del modelo a lo largo de los avances (F1 en el conjunto de prueba)
evolucion = pd.DataFrame({
    "Etapa": ["Avance II\nGB por defecto", "Avance III\nGB ajustado", "Avance IV\nModelo A final", "Avance IV\nModelo B (sin likes)"],
    "F1": [0.9402, 0.9459, resultados.loc[0, "F1"], resultados.loc[1, "F1"]]})
plt.figure(figsize=(8, 4))
barras = plt.bar(evolucion["Etapa"], evolucion["F1"], color=["#9aa5b1", "#4c72b0", "#2f4b7c", "#dd8452"])
for b, v in zip(barras, evolucion["F1"]):
    plt.text(b.get_x() + b.get_width() / 2, v + 0.005, f"{v:.4f}", ha="center")
plt.ylim(0.75, 1.0); plt.ylabel("F1-score en prueba"); plt.title("Evolución del desempeño por avance")
plt.tight_layout(); plt.savefig("evolucion_avances.png", dpi=150); plt.show()

In [ ]:
# Análisis de errores por categoría (palabra clave) del modelo A
analisis = X_test[["Keyword"]].copy()
analisis["real"] = y_test.values
analisis["pred_A"] = modelo_A.predict(X_test)
analisis["pred_B"] = modelo_B.predict(X_test)
errores = (analisis.assign(error_A=lambda d: d.real != d.pred_A, error_B=lambda d: d.real != d.pred_B)
           .groupby("Keyword").agg(videos=("real", "size"), error_A=("error_A", "mean"), error_B=("error_B", "mean"))
           .sort_values("error_B", ascending=False))
errores.to_csv("errores_por_categoria.csv")
errores.head(10).round(3)

## 7. Persistencia de los modelos
Se guardan los pipelines completos (preprocesamiento + modelo) y sus metadatos para versionarlos en GitHub.

In [ ]:
os.makedirs("modelos", exist_ok=True)
joblib.dump(modelo_A, "modelos/modelo_A_gradient_boosting.joblib")
joblib.dump(modelo_B, "modelos/modelo_B_xgboost_tfidf.joblib")
metadatos = {"version": "1.0.0", "fecha_entrenamiento": datetime.now().strftime("%Y-%m-%d %H:%M"),
             "mediana_visualizaciones": float(MEDIANA_VISTAS), "fecha_referencia": str(FECHA_REFERENCIA.date()),
             "variables_A": VARS_A, "variables_B": VARS_B,
             "metricas": resultados.drop(columns="Entrenamiento_s").round(4).to_dict(orient="records")}
with open("modelos/metadatos.json", "w", encoding="utf-8") as f:
    json.dump(metadatos, f, ensure_ascii=False, indent=2)
resultados.to_csv("resultados_finales.csv", index=False)
print(os.listdir("modelos"))

## 8. Función de predicción
Recibe los datos de un video tal como los vería un usuario y construye internamente las mismas variables del entrenamiento.

In [ ]:
CATEGORIAS = sorted(datos["Keyword"].unique())

def construir_registro(titulo, categoria, fecha_publicacion, likes, comentarios_video, texto_comentarios,
                       sentimientos=None, likes_comentarios=None):
    lista = [c.strip() for c in str(texto_comentarios).split("\n") if c.strip()] or [""]
    n = len(lista)
    sent = sentimientos if sentimientos is not None else [1] * n
    lk = likes_comentarios if likes_comentarios is not None else [0] * n
    fecha = pd.to_datetime(fecha_publicacion)
    titulo = str(titulo)
    r = {"Title": titulo, "Keyword": categoria, "texto_comentarios": " ".join(lista),
         "comentarios_analizados": n, "log_promedio_likes_comentarios": np.log1p(np.mean(lk)),
         "proporcion_negativos": np.mean([s == 0 for s in sent]),
         "proporcion_neutrales": np.mean([s == 1 for s in sent]),
         "proporcion_positivos": np.mean([s == 2 for s in sent]),
         "longitud_titulo": len(titulo), "antiguedad_dias": max((FECHA_REFERENCIA - fecha).days, 0),
         "log_likes_video": np.log1p(likes) if likes is not None else np.nan,
         "log_comentarios_video": np.log1p(comentarios_video) if comentarios_video is not None else np.nan,
         "mayusculas_titulo": sum(c.isupper() for c in titulo) / max(len(titulo), 1),
         "signos_titulo": sum(titulo.count(s) for s in "!?"), "palabras_titulo": len(titulo.split()),
         "titulo_numero": int(any(c.isdigit() for c in titulo)),
         "longitud_comentarios": len(" ".join(lista)) / n}
    return pd.DataFrame([r])

def predecir(titulo, categoria, fecha_publicacion, texto_comentarios, likes=None, comentarios_video=None):
    registro = construir_registro(titulo, categoria, fecha_publicacion, likes, comentarios_video, texto_comentarios)
    usar_A = likes is not None and comentarios_video is not None
    modelo, nombre = (modelo_A, "Modelo A (con likes)") if usar_A else (modelo_B, "Modelo B (NLP sin likes)")
    prob = float(modelo.predict_proba(registro)[0, 1])
    salida = {"modelo": nombre, "probabilidad_alta": round(prob, 4),
              "prediccion": "Popularidad ALTA" if prob >= 0.5 else "Popularidad BAJA"}
    registrar_prediccion(registro, salida)
    return salida

REGISTRO = "monitoreo_predicciones.csv"
def registrar_prediccion(registro, salida):
    fila = registro.drop(columns=["texto_comentarios"]).assign(fecha_prediccion=datetime.now().isoformat(timespec="seconds"), **salida)
    fila.to_csv(REGISTRO, mode="a", header=not os.path.exists(REGISTRO), index=False)

# Prueba con un video real del conjunto de prueba
ejemplo = X_test.iloc[0]
print("Video:", ejemplo["Title"][:80], "| Real:", "Alta" if y_test.iloc[0] else "Baja")
print(predecir(ejemplo["Title"], ejemplo["Keyword"], ejemplo["Published At"],
               ejemplo["texto_comentarios"], ejemplo["Likes"], ejemplo["Comments"]))
print(predecir(ejemplo["Title"], ejemplo["Keyword"], ejemplo["Published At"], ejemplo["texto_comentarios"]))

## 9. Interfaz web y API con Gradio
Gradio publica una interfaz para el usuario final y, de forma automática, un endpoint de API (`/predecir`). En Colab se genera un enlace público temporal (`share=True`).

In [ ]:
import gradio as gr

def interfaz(titulo, categoria, fecha, comentarios_texto, usar_metricas, likes, comentarios_video):
    if not titulo or not comentarios_texto:
        return "Ingrese el título y al menos un comentario.", None
    r = predecir(titulo, categoria, fecha, comentarios_texto,
                 likes if usar_metricas else None, comentarios_video if usar_metricas else None)
    texto = f"### {r['prediccion']}\n**{r['modelo']}** · probabilidad de popularidad alta: **{r['probabilidad_alta']:.1%}**"
    return texto, {"Alta": r["probabilidad_alta"], "Baja": 1 - r["probabilidad_alta"]}

with gr.Blocks(title="Predictor de popularidad YouTube") as demo:
    gr.Markdown("# Predictor de popularidad de videos de YouTube\nProyecto de PLN · Universidad de Cundinamarca. "
                "Escriba los comentarios del video (uno por línea). Si conoce los likes y comentarios del video se usa el Modelo A; si no, el Modelo B basado en texto.")
    with gr.Row():
        with gr.Column():
            titulo = gr.Textbox(label="Título del video")
            categoria = gr.Dropdown(CATEGORIAS, value="tech", label="Categoría (palabra clave)")
            fecha = gr.Textbox(label="Fecha de publicación (AAAA-MM-DD)", value="2022-08-01")
            comentarios_texto = gr.Textbox(label="Comentarios (uno por línea)", lines=6)
            usar_metricas = gr.Checkbox(label="Conozco los likes y comentarios del video", value=False)
            likes = gr.Number(label="Likes del video", value=1000)
            comentarios_video = gr.Number(label="Comentarios del video", value=100)
            boton = gr.Button("Predecir", variant="primary")
        with gr.Column():
            resultado = gr.Markdown()
            grafico = gr.Label(label="Probabilidad por clase")
    boton.click(interfaz, [titulo, categoria, fecha, comentarios_texto, usar_metricas, likes, comentarios_video],
                [resultado, grafico], api_name="predecir")

demo.launch(share=EN_COLAB, prevent_thread_lock=True, server_port=7860)

In [ ]:
# Consumo del modelo como API usando el cliente oficial de Gradio
from gradio_client import Client
cliente = Client("http://127.0.0.1:7860/", verbose=False)
respuesta = cliente.predict("My New House Gaming Setup is SICK!", "tech", "2022-08-23",
                            "This setup is insane, love it!\nBest video this year\nWhere did you buy that desk?",
                            False, 0, 0, api_name="/predecir")
print(respuesta[0])

## 10. Monitoreo del modelo
Se verifican tres aspectos: (1) registro de cada predicción, (2) deriva de datos mediante el índice de estabilidad poblacional (PSI) y (3) alerta de desempeño si el F1 de un lote etiquetado cae por debajo de 0,90 (Modelo A) o 0,80 (Modelo B).

In [ ]:
def psi(referencia, actual, cortes=10):
    bordes = np.unique(np.quantile(referencia, np.linspace(0, 1, cortes + 1)))
    bordes[0], bordes[-1] = -np.inf, np.inf
    ref = np.histogram(referencia, bordes)[0] / len(referencia) + 1e-6
    act = np.histogram(actual, bordes)[0] / len(actual) + 1e-6
    return float(np.sum((act - ref) * np.log(act / ref)))

# Lote "en producción": conjunto de prueba (videos no vistos) y un lote simulado más reciente
lote_reciente = X_test.copy()
lote_reciente["antiguedad_dias"] = (lote_reciente["antiguedad_dias"] * 0.4).round()  # videos más nuevos
filas = []
for var in ["log_likes_video", "log_comentarios_video", "antiguedad_dias", "proporcion_positivos", "longitud_titulo"]:
    for nombre_lote, lote in [("Prueba", X_test), ("Simulado reciente", lote_reciente)]:
        v = psi(X_train[var], lote[var])
        filas.append({"Variable": var, "Lote": nombre_lote, "PSI": round(v, 4),
                      "Estado": "Estable" if v < 0.1 else ("Vigilar" if v < 0.25 else "Deriva: reentrenar")})
monitoreo = pd.DataFrame(filas)
monitoreo.to_csv("monitoreo_psi.csv", index=False)
display(monitoreo.pivot(index="Variable", columns="Lote", values="Estado"))

UMBRALES = {"A": 0.90, "B": 0.80}
for clave, modelo in [("A", modelo_A), ("B", modelo_B)]:
    f1_lote = f1_score(y_test, modelo.predict(X_test))
    print(f"Modelo {clave}: F1 del lote = {f1_lote:.4f} ->", "OK" if f1_lote >= UMBRALES[clave] else "ALERTA: revisar/reentrenar")
print("\nPredicciones registradas:", len(pd.read_csv(REGISTRO)))

## 11. Conclusiones del producto final

- El proyecto termina con **dos modelos complementarios** versionados: el Modelo A (Gradient Boosting) para clasificar la popularidad cuando se conocen las métricas de interacción y el Modelo B (XGBoost + TF-IDF), que estima la popularidad a partir del texto de comentarios y del título, sin usar likes.
- El modelo es **consumible por un usuario** mediante una interfaz web y una API de Gradio, ejecutable en la nube desde Google Colab.
- Se implementó un **monitoreo básico** con registro de predicciones, PSI para deriva de datos y umbrales de desempeño que indican cuándo reentrenar.